# Day 5 - Topic 01: Data & Population
### Module: Data Analytics / Statistics Foundations

---

## 1. Population vs. Sample
In statistics, the distinction between a **Population** and a **Sample** is foundational:

| Attribute | Population | Sample |
| :--- | :--- | :--- |
| **Definition** | The complete collection of all elements/observations of interest. | A representative subset drawn from the population. |
| **Size Notation** | $N$ (Total population size) | $n$ (Sample size, $n \ll N$) |
| **Measures** | **Parameters** (Greek letters: $\mu, \sigma^2, \sigma, P$) | **Statistics** (Latin letters: $\bar{x}, s^2, s, \hat{p}$) |
| **Mean** | $\mu = \frac{1}{N} \sum_{i=1}^N x_i$ | $\bar{x} = \frac{1}{n} \sum_{i=1}^n x_i$ |
| **Variance** | $\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2$ (`ddof=0`) | $s^2 = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2$ (`ddof=1`, Bessel's correction) |
| **Feasibility** | Often impossible or expensive to measure completely. | Cost-effective, fast, and practical. |

---

## 2. Sampling Error vs. Sampling Bias
- **Sampling Error:** Natural variability between a sample statistic and the true population parameter: $\text{Sampling Error} = \bar{x} - \mu$.
  - Regulated by the **Standard Error of the Mean (SE)**: $\text{SE} = \frac{\sigma}{\sqrt{n}}$. As $n \uparrow$, $\text{SE} \downarrow$.
- **Sampling Bias:** Systematic flaw in sampling that causes certain groups to be over-represented or under-represented (e.g. voluntary response bias, survivorship bias).

In [1]:
# 1. Create a known population of N = 10,000 observations
import numpy as np
import pandas as pd

np.random.seed(42)
N_pop = 10_000
population = np.random.gamma(shape=5.0, scale=12.0, size=N_pop) + 20.0

# Calculate True Population Parameters (divide by N, ddof=0)
mu_pop = np.mean(population)
var_pop = np.var(population, ddof=0)
std_pop = np.std(population, ddof=0)

print("=" * 65)
print(f"TRUE POPULATION PARAMETERS (N = {N_pop:,})")
print("=" * 65)
print(f"Population Mean (μ)         : {mu_pop:8.2f}")
print(f"Population Variance (σ²)    : {var_pop:8.2f}")
print(f"Population Std Dev (σ)      : {std_pop:8.2f}")

# 2. Draw samples of varying sizes and observe convergence (Law of Large Numbers)
sample_sizes = [10, 50, 100, 500, 1000, 5000]
records = []

for n in sample_sizes:
    sample = np.random.choice(population, size=n, replace=False)
    x_bar = np.mean(sample)
    s_var = np.var(sample, ddof=1)  # Unbiased sample variance
    records.append({
        'Sample Size (n)': n,
        'Sample Mean (x̄)': round(x_bar, 2),
        'Sampling Error (x̄ - μ)': round(x_bar - mu_pop, 2),
        'Sample Var (s², ddof=1)': round(s_var, 2),
        'Standard Error (σ/√n)': round(std_pop / np.sqrt(n), 2)
    })

df_res = pd.DataFrame(records)
print("\nSAMPLE STATISTICS VS. POPULATION PARAMETERS:")
print(df_res.to_string(index=False))


TRUE POPULATION PARAMETERS (N = 10,000)
Population Mean (μ)         :    80.43
Population Variance (σ²)    :   709.07
Population Std Dev (σ)      :    26.63

SAMPLE STATISTICS VS. POPULATION PARAMETERS:
 Sample Size (n)  Sample Mean (x̄)  Sampling Error (x̄ - μ)  Sample Var (s², ddof=1)  Standard Error (σ/√n)
              10             77.76                    -2.67                   483.40                   8.42
              50             80.14                    -0.29                   586.41                   3.77
             100             80.14                    -0.28                   617.93                   2.66
             500             78.69                    -1.74                   624.01                   1.19
            1000             80.78                     0.36                   781.10                   0.84
            5000             80.79                     0.37                   751.02                   0.38


## 3. Four Major Probability Sampling Techniques

1. **Simple Random Sampling (SRS):** Every member in the population has an equal probability of selection ($P = 1/N$).
2. **Systematic Sampling:** Every $k$-th member is selected from an ordered list after a random starting point ($k = \lfloor N / n \rfloor$).
3. **Stratified Random Sampling:** The population is divided into mutually exclusive, homogeneous groups (**strata**, e.g., Department, Gender, Income level), and random samples are drawn proportionally from each stratum.
4. **Cluster Sampling:** The population is naturally grouped into heterogeneous **clusters** (e.g., city branches, schools). Random clusters are selected, and members inside those clusters are sampled.

In [2]:
# Demonstration of the 4 Sampling Techniques using a corporate population (N=1000)
N_org = 1_000
departments = np.random.choice(
    ['Engineering', 'Sales', 'Finance', 'HR'],
    size=N_org,
    p=[0.45, 0.30, 0.15, 0.10]
)

base_salaries = {'Engineering': 110, 'Sales': 85, 'Finance': 95, 'HR': 70}
salaries = [np.random.normal(base_salaries[d], 15) for d in departments]

df_pop = pd.DataFrame({
    'emp_id': range(1, N_org + 1),
    'department': departments,
    'salary': np.round(salaries, 2),
    'branch': np.random.choice(['Pune', 'Bengaluru', 'Hyderabad'], size=N_org)
})

sample_n = 100

# 1. Simple Random Sample
srs = df_pop.sample(n=sample_n, random_state=42)

# 2. Systematic Sample (every k-th item)
k = N_org // sample_n
start = np.random.randint(0, k)
systematic = df_pop.iloc[start::k]

# 3. Stratified Sample (preserves department proportions)
stratified = df_pop.groupby('department', group_keys=False).sample(frac=sample_n / N_org, random_state=42)

# 4. Cluster Sample (pick 1 branch)
chosen_branch = np.random.choice(df_pop['branch'].unique())
cluster = df_pop[df_pop['branch'] == chosen_branch].sample(n=sample_n, random_state=42)

print(f"True Population Mean Salary : {df_pop['salary'].mean():.2f} k")
print(f"SRS Sample Mean Salary      : {srs['salary'].mean():.2f} k (Error: {srs['salary'].mean() - df_pop['salary'].mean():+.2f})")
print(f"Systematic Sample Mean      : {systematic['salary'].mean():.2f} k (Error: {systematic['salary'].mean() - df_pop['salary'].mean():+.2f})")
print(f"Stratified Sample Mean      : {stratified['salary'].mean():.2f} k (Error: {stratified['salary'].mean() - df_pop['salary'].mean():+.2f})")
print(f"Cluster ({chosen_branch}) Mean    : {cluster['salary'].mean():.2f} k (Error: {cluster['salary'].mean() - df_pop['salary'].mean():+.2f})")


True Population Mean Salary : 96.15 k
SRS Sample Mean Salary      : 94.11 k (Error: -2.04)
Systematic Sample Mean      : 95.10 k (Error: -1.05)
Stratified Sample Mean      : 95.86 k (Error: -0.29)
Cluster (Hyderabad) Mean    : 96.71 k (Error: +0.57)


---
# 4. Synthetic Data Generation: Concepts, Types & Methods

### What is Synthetic Data?
**Synthetic Data** is artificially generated information manufactured through mathematical models, probabilistic distributions, or generative machine learning algorithms, rather than collected from direct real-world measurements. It preserves the statistical properties, correlations, and distributions of real data while containing no real individual records.

### Why is Synthetic Data Critical in Data Analytics?
1. **Privacy Preservation & Regulatory Compliance:** Comply with strict privacy laws (**GDPR, HIPAA, DPDP Act 2023**) by eliminating Personally Identifiable Information (PII) before sharing with external vendors or research teams.
2. **Mitigating Extreme Class Imbalance:** In fraud detection ($< 0.1\%$ fraud) or rare disease diagnosis, synthetic oversampling provides balanced training data.
3. **Data Scarcity & Cold-Start Problems:** Simulating customer behavioral paths before launching a new product or application.
4. **Stress-Testing & Edge-Case Simulation:** Testing autonomous vehicles, trading systems, or credit risk pipelines against extreme "black swan" market scenarios.
5. **Cost & Speed:** Collecting and labeling $100,000$ medical images or survey responses can take months and millions of rupees; synthetic generation takes seconds.

---

### The 5 Major Paradigms / Types of Synthetic Data Generation

| Paradigm | Technique / Method | Best Suited For | Real-World Example |
| :--- | :--- | :--- | :--- |
| **1. Rule-Based / Deterministic** | Domain rules, regex patterns, lookup tables, business schemas | Mocking relational databases, mock APIs | Generating synthetic names, addresses, credit card numbers, GSTINs via `Faker`. |
| **2. Parametric Statistical Distributions** | Fitting theoretical PDFs (Normal, Poisson, Exponential, Gamma) | Scientific modeling, queue simulations | Modeling call-center wait times with Exponential distribution, footfall with Poisson. |
| **3. Non-Parametric Resampling** | Bootstrapping, empirical KDE sampling, jittering (noise perturbation) | When the underlying distribution is unknown or multimodal | Sampling historical retail sales with replacement + adding 2% Gaussian noise. |
| **4. Machine Learning-Based Generators** | **SMOTE**, `sklearn.datasets` (`make_classification`, `make_regression`, `make_blobs`) | Machine learning benchmarking, balancing imbalanced datasets | Synthesizing borderline minority credit defaults along $k$-nearest neighbors. |
| **5. Deep Generative Models** | **CTGAN** (Conditional Tabular GAN), Variational Autoencoders (VAEs), Diffusion | High-dimensional complex tabular data with non-linear relationships | Enterprise synthetic data platforms (SDV, Gretel, Mostly.ai). |

In [3]:
# ==============================================================================
# Type 2: Generating Synthetic Data from Theoretical Distributions
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
N_samples = 5_000

# 1. Normal (Gaussian): Heights of adult males (mean = 175 cm, std = 7 cm)
synth_normal = np.random.normal(loc=175, scale=7, size=N_samples)

# 2. Uniform: Unbiased lottery draws / random wait duration [10 to 60 minutes]
synth_uniform = np.random.uniform(low=10, high=60, size=N_samples)

# 3. Exponential: Customer service waiting time / server response latency (scale = 1/lambda = 8 mins)
synth_exponential = np.random.exponential(scale=8.0, size=N_samples)

# 4. Poisson: Number of customer arrivals or website transactions per hour (lambda = 15 events)
synth_poisson = np.random.poisson(lam=15, size=N_samples)

# 5. Binomial: Number of successful ad conversions out of n=50 clicks (conversion rate p = 0.08)
synth_binomial = np.random.binomial(n=50, p=0.08, size=N_samples)

# Summary table of generated synthetic distributions
dist_summary = pd.DataFrame([
    {'Distribution': 'Normal (Gaussian)', 'Target Use Case': 'Heights, measurement errors', 'Theoretical Mean': 175.0, 'Sample Mean': round(np.mean(synth_normal), 2), 'Sample Std': round(np.std(synth_normal), 2)},
    {'Distribution': 'Uniform', 'Target Use Case': 'Bounded simulation, baseline', 'Theoretical Mean': 35.0, 'Sample Mean': round(np.mean(synth_uniform), 2), 'Sample Std': round(np.std(synth_uniform), 2)},
    {'Distribution': 'Exponential', 'Target Use Case': 'Waiting times, equipment failure', 'Theoretical Mean': 8.0, 'Sample Mean': round(np.mean(synth_exponential), 2), 'Sample Std': round(np.std(synth_exponential), 2)},
    {'Distribution': 'Poisson', 'Target Use Case': 'Event counts per time window', 'Theoretical Mean': 15.0, 'Sample Mean': round(np.mean(synth_poisson), 2), 'Sample Std': round(np.std(synth_poisson), 2)},
    {'Distribution': 'Binomial', 'Target Use Case': 'Ad conversions out of n=50 trials', 'Theoretical Mean': 4.0, 'Sample Mean': round(np.mean(synth_binomial), 2), 'Sample Std': round(np.std(synth_binomial), 2)}
])

print("=" * 85)
print("           SYNTHETIC DATA GENERATION: THEORETICAL DISTRIBUTIONS SUMMARY          ")
print("=" * 85)
print(dist_summary.to_string(index=False))


           SYNTHETIC DATA GENERATION: THEORETICAL DISTRIBUTIONS SUMMARY          
     Distribution                   Target Use Case  Theoretical Mean  Sample Mean  Sample Std
Normal (Gaussian)       Heights, measurement errors             175.0       175.04        6.97
          Uniform      Bounded simulation, baseline              35.0        34.74       14.28
      Exponential  Waiting times, equipment failure               8.0         8.10        8.27
          Poisson      Event counts per time window              15.0        15.00        3.79
         Binomial Ad conversions out of n=50 trials               4.0         3.99        1.92


## 5. Generating Correlated Tabular Data (Multivariate Normal)

In real-world data analytics, individual variables rarely exist in isolation; they exhibit **inter-feature correlations**:
- Experience is strongly correlated with Salary ($r \approx 0.85$).
- House Square Footage is correlated with Price ($r \approx 0.80$).

To synthesize realistic multidimensional tables, we compute a **Covariance Matrix** $\mathbf{\Sigma}$ from the desired standard deviations $\mathbf{D}$ and correlation matrix $\mathbf{R}$:
$$\mathbf{\Sigma} = \mathbf{D} \cdot \mathbf{R} \cdot \mathbf{D}$$
and draw vectors using `np.random.multivariate_normal(mean_vector, cov_matrix, size=N)`.

In [4]:
# ==============================================================================
# Generating Multi-Feature Correlated Tabular Data
# Features: [Age (years), Experience (years), Annual Salary (INR Lakhs)]
# ==============================================================================
N = 2_000

# Desired Mean Vector: [Age, Experience, Salary_Lakhs]
mean_vec = np.array([32.0, 8.0, 16.0])

# Desired Standard Deviations: [Age: 6 yrs, Experience: 5 yrs, Salary: 7 Lakhs]
stds = np.array([6.0, 5.0, 7.0])

# Target Correlation Matrix (R)
corr_matrix = np.array([
    [1.00, 0.88, 0.80],  # Age vs [Age, Exp, Salary]
    [0.88, 1.00, 0.92],  # Exp vs [Age, Exp, Salary]  (Strong correlation between exp and salary!)
    [0.80, 0.92, 1.00]   # Salary vs [Age, Exp, Salary]
])

# Convert Correlation Matrix to Covariance Matrix: Cov = diag(stds) * Corr * diag(stds)
D = np.diag(stds)
cov_matrix = D @ corr_matrix @ D

# Generate Synthetic Multi-Feature Observations
raw_synth = np.random.multivariate_normal(mean_vec, cov_matrix, size=N)

df_synthetic_emp = pd.DataFrame(raw_synth, columns=['age', 'experience', 'salary_lakhs'])
# Post-processing domain constraints (e.g. minimum age 21, non-negative experience)
df_synthetic_emp['age'] = df_synthetic_emp['age'].clip(lower=21).round(1)
df_synthetic_emp['experience'] = np.clip(df_synthetic_emp['experience'], 0, df_synthetic_emp['age'] - 20).round(1)
df_synthetic_emp['salary_lakhs'] = df_synthetic_emp['salary_lakhs'].clip(lower=3.5).round(2)

print("SYNTHETIC MULTIVARIATE DATASET PREVIEW (First 5 records):")
print(df_synthetic_emp.head())

print("\nVERIFYING RECOVERED CORRELATION MATRIX (Close to target!):")
print(df_synthetic_emp.corr().round(3))


SYNTHETIC MULTIVARIATE DATASET PREVIEW (First 5 records):
    age  experience  salary_lakhs
0  36.8        11.0         19.76
1  34.4         8.2         15.85
2  31.0        10.4         19.40
3  40.1        16.1         25.18
4  34.9         9.2         20.12

VERIFYING RECOVERED CORRELATION MATRIX (Close to target!):
                age  experience  salary_lakhs
age           1.000       0.902         0.814
experience    0.902       1.000         0.917
salary_lakhs  0.814       0.917         1.000


## 6. Type 4: Machine Learning & SMOTE Oversampling

### Synthetic Minority Over-sampling Technique (SMOTE - Chawla et al., 2002)
When real datasets have severe class imbalance (e.g. $95\%$ non-fraud vs $5\%$ fraud), simple duplication causes overfitting. **SMOTE** synthesizes *brand new, interpolated data points* between existing minority samples:

1. For each minority class sample $\vec{x}_i$, find its $k$ nearest neighbors in the same class.
2. Randomly select one neighbor $\vec{x}_{zi}$.
3. Synthesize a new sample along the connecting line segment:
   $$\vec{x}_{\text{new}} = \vec{x}_i + \lambda \cdot (\vec{x}_{zi} - \vec{x}_i) \quad \text{where } \lambda \sim \text{Uniform}(0, 1)$$

In addition, Scikit-Learn provides built-in generators for benchmarking:
- `make_classification`: Generate multi-class tabular classification data.
- `make_regression`: Generate linear/non-linear targets with controllable noise.
- `make_blobs`: Generate spherical Gaussian clusters for unsupervised clustering.

In [5]:
# ==============================================================================
# SMOTE Implementation from Scratch & Imbalanced Dataset Generation
# ==============================================================================
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors

# 1. Create an imbalanced synthetic classification dataset (92% Class 0, 8% Class 1)
X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.92, 0.08],
    random_state=42
)

n_majority = np.sum(y == 0)
n_minority = np.sum(y == 1)
print(f"Original Class Distribution: Class 0 = {n_majority} (Majority) | Class 1 = {n_minority} (Minority)")

# 2. Vectorized SMOTE Implementation from First Principles
def custom_smote(X_minority, n_samples_needed, k_neighbors=5):
    knn = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_minority)
    _, indices = knn.kneighbors(X_minority)
    
    synthetic_points = []
    for _ in range(n_samples_needed):
        # Pick a random minority point
        idx = np.random.randint(0, len(X_minority))
        # Pick a random neighbor from its k-nearest neighbors (excluding itself at index 0)
        neighbor_idx = indices[idx, np.random.randint(1, k_neighbors + 1)]
        
        # Interpolate along the vector
        diff = X_minority[neighbor_idx] - X_minority[idx]
        lam = np.random.uniform(0, 1)
        synthetic_points.append(X_minority[idx] + lam * diff)
        
    return np.array(synthetic_points)

n_to_synthesize = n_majority - n_minority
X_minority = X[y == 1]
X_synthetic_minority = custom_smote(X_minority, n_samples_needed=n_to_synthesize, k_neighbors=5)

# Combine real majority + real minority + synthetic minority
X_balanced = np.vstack([X, X_synthetic_minority])
y_balanced = np.hstack([y, np.ones(n_to_synthesize, dtype=int)])

print(f"Synthesized Minority Samples : {len(X_synthetic_minority)}")
print(f"Balanced Class Distribution  : Class 0 = {np.sum(y_balanced == 0)} | Class 1 = {np.sum(y_balanced == 1)}")


Original Class Distribution: Class 0 = 915 (Majority) | Class 1 = 85 (Minority)
Synthesized Minority Samples : 830
Balanced Class Distribution  : Class 0 = 915 | Class 1 = 915


## 7. Synthetic Data Evaluation: Fidelity, Correlation & Utility

How do data analytics professionals evaluate synthetic data quality?
1. **Statistical Fidelity (Distribution Matching):**
   - **Two-Sample Kolmogorov-Smirnov (KS) Test:** Tests whether real and synthetic features originate from the exact same continuous distribution. ($p$-value $> 0.05$ indicates statistical indistinguishability).
2. **Correlation Matrix Preservation:**
   - Quantified by the **Frobenius Norm Distance**: $\|\mathbf{R}_{\text{real}} - \mathbf{R}_{\text{synth}}\|_{F} \rightarrow 0$.
3. **Machine Learning Utility (TSTR):**
   - **Train on Synthetic, Test on Real (TSTR):** Train an ML model exclusively on synthetic data, and test on held-out real data. If accuracy and F1 scores are comparable, the data has high utility.

In [6]:
# ==============================================================================
# Statistical Fidelity Assessment: Two-Sample KS Test
# ==============================================================================
# Real baseline sample from earlier population
real_data = np.random.gamma(shape=5.0, scale=12.0, size=1000) + 20.0
# Good synthetic data (correctly fitted)
synth_good = np.random.gamma(shape=5.0, scale=12.0, size=1000) + 20.0
# Poor synthetic data (wrong distribution / underestimating spread)
synth_poor = np.random.normal(loc=np.mean(real_data), scale=10.0, size=1000)

ks_good = stats.ks_2samp(real_data, synth_good)
ks_poor = stats.ks_2samp(real_data, synth_poor)

print("=" * 75)
print("       TWO-SAMPLE KOLMOGOROV-SMIRNOV (KS) FIDELITY TEST EVALUATION       ")
print("=" * 75)
print(f"Well-Fitted Synthetic Data : KS Statistic = {ks_good.statistic:.4f}, p-value = {ks_good.pvalue:.4f}")
print(f"  => Assessment: {'Indistinguishable from Real (Passed!)' if ks_good.pvalue > 0.05 else 'Distinguishable'}")

print(f"\nPoorly-Fitted Synthetic Data: KS Statistic = {ks_poor.statistic:.4f}, p-value = {ks_poor.pvalue:.4e}")
print(f"  => Assessment: {'Indistinguishable' if ks_poor.pvalue > 0.05 else 'FAILED (Statistically different from Real data!)'}")


       TWO-SAMPLE KOLMOGOROV-SMIRNOV (KS) FIDELITY TEST EVALUATION       
Well-Fitted Synthetic Data : KS Statistic = 0.0340, p-value = 0.6102
  => Assessment: Indistinguishable from Real (Passed!)

Poorly-Fitted Synthetic Data: KS Statistic = 0.2800, p-value = 6.4654e-35
  => Assessment: FAILED (Statistically different from Real data!)
